In [ ]:
#r "BoSSSpad.dll"
using System;
using System.Data;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using System.Diagnostics;
using Microsoft.AspNetCore.Html;
using System.Text.RegularExpressions;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.Timestepping;
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using BoSSS.Solution.LevelSetTools.PhasefieldLevelSet;

Init();

A static droplet with two equal fluids is placed in a closed box.  
This testcase demonstrates the discrepancy between the fluid solver equilibrium contact angle  
and the phasefield level set equilibrium contact angle.  
Only as the interface thickness tends towards zero the latter value aproaches the first one.  
Due to this discrepancy parasitic currents are excited.  

The material values are chosen to be  
PhasefieldContactLine : $\rho = 1000, \sigma = 1.0, M = 1, \Delta t = 0.001$  
The cahn number is varied.  
The experiment is repeated for different prescribed static contact angles.

In [ ]:
string proj = "PhasefieldContactLine";
BoSSSshell.WorkflowMgm.Init(proj);
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();


In [ ]:
static XNSE_Control BoxStaticDropletWall(int p = 2, int kelem = 10, int amrlevel = 2, double diff = 1.0, double cahnfactor = 1.0, double theta = Math.PI/4.0, LevelSetEvolution evo = LevelSetEvolution.StokesExtension) {

    XNSE_Control C = new XNSE_Control();

    // basic database options
    // ======================
    C.savetodb = true;
    C.SetDatabase(BoSSSshell.WorkflowMgm.DefaultDatabase);
    C.ProjectName = "PhasefieldContactLine";
    C.ProjectDescription = "static droplet contact angle test";

    C.PostprocessingModules.Add(new BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases.MovingContactLineLogging());
    C.PostprocessingModules.Add(new BoSSS.Application.XNSE_Solver.Logging.EnergyLogging());

    // DG degrees
    // ==========
    C.SetDGdegree(p);

    // Physical Parameters
    // ===================
    C.PhysicalParameters.rho_A = 1000;
    C.PhysicalParameters.rho_B = 1000;
    C.PhysicalParameters.mu_A = 1;
    C.PhysicalParameters.mu_B = 1;
    C.PhysicalParameters.Sigma = 1.0;
    C.PhysicalParameters.theta_e = theta;

    C.PhysicalParameters.betaS_A = 0.0;
    C.PhysicalParameters.betaS_B = 0.0;

    C.PhysicalParameters.IncludeConvection = true;
    C.PhysicalParameters.Material = false;

    // grid generation
    // ===============

    double h = 1.0 / (double)kelem;
    C.GridFunc = delegate () {
        double[] Xnodes = GenericBlas.Linspace(-1, 1, 2 * kelem + 1);
        double[] Ynodes = GenericBlas.Linspace(0, 1, kelem + 1);
        var grd = Grid2D.Cartesian2DGrid(Xnodes, Ynodes, periodicX: false);
    
        grd.EdgeTagNames.Add(1, "NavierSlip_linear");
    
        grd.DefineEdgeTags(delegate (double[] X) {
            return 1;
        });
    
        return grd;
    };
    
    // boundary conditions
    // ===================
    C.AddBoundaryValue("NavierSlip_linear");

    // Initial Values
    // ==============
    double radius = 0.5;    
    double[] center = new double[] { 0.0, -radius * Math.Cos(theta) };

    C.Option_LevelSetEvolution = evo;
    C.Tags.Add(evo.ToString());
    Formula PhiFunc;
    double cahn = cahnfactor * 4 * 1.0/(kelem * Math.Pow(2.0, amrlevel)) / (2 * p + 1);
    PhiFunc = new Formula($"X => Math.Tanh((((X[0] - {center[0]}).Pow2() + (X[1] - {center[1]}).Pow2()).Sqrt() - {radius})/(Math.Sqrt(2)*{cahn}))"); // signed-distance form
    C.PhasefieldControl = new PhasefieldSettings() {
        cahn = cahn,
        diff = diff,
        TimeSteppingScheme = TimeSteppingScheme.ImplicitEuler,
        CorrectionType = BoSSS.Solution.LevelSetTools.PhasefieldLevelSet.Correction.None,
        theta = $"X => X[1] < 1e-8 ? {theta} : Math.PI/2.0"
    };       
    C.AddInitialValue("Phi", PhiFunc);

    // misc. solver options
    // ====================
    C.LSContiProjectionMethod = ContinuityProjectionOption.ConstrainedDG;

    C.NonLinearSolver.MaxSolverIterations = 20;
    C.FailOnSolverFail = false;
    C.NonLinearSolver.ConvergenceCriterion = 1e-8;
    C.LevelSet_ConvergenceCriterion = 1e-6;

    C.AdvancedDiscretizationOptions.SST_isotropicMode = BoSSS.Solution.LevelSetTools.SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;

    C.AdaptiveMeshRefinement = amrlevel > 0;
    C.activeAMRlevelIndicators.Add(new AMRonNarrowband() { maxRefinementLevel = amrlevel });
    C.AMR_startUpSweeps = amrlevel;

    // Timestepping
    // ============
    C.TimeSteppingScheme = TimeSteppingScheme.ImplicitEuler;
    C.Timestepper_BDFinit = TimeStepperInit.SingleInit;
    C.Timestepper_LevelSetHandling = LevelSetHandling.LieSplitting;

    C.TimesteppingMode = AppControl._TimesteppingMode.Transient;

    double dt = 0.001;
    C.dtMax = dt;
    C.dtMin = dt;
    C.NoOfTimesteps = 1000;
    C.Endtime = 1.0;
    C.saveperiod = 1;
    
    C.SessionName = "ContactAngle_" + theta.ToString("N4") + "_AMR" + amrlevel + "_P" + p + "_" + evo.ToString() + (evo == LevelSetEvolution.Phasefield ? "_" + C.PhasefieldControl.diff.ToString("N4") + "_" + C.PhasefieldControl.cahn.ToString("N4") : "");
    C.TracingNamespaces = "BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater";
    
    return C;
}

In [204]:
double[] thetaS = new double[] {45.0, 60.0, 75.0, 90.0};
List<XNSE_Control> controls = new List<XNSE_Control>();

In [205]:
foreach(double thetaDeg in thetaS){
    double theta = thetaDeg / 180.0 * Math.PI;
    controls.Add(BoxStaticDropletWall(evo:LevelSetEvolution.StokesExtension, theta: theta));
    controls.Add(BoxStaticDropletWall(evo:LevelSetEvolution.Phasefield, cahnfactor: 1.0, theta: theta));
    controls.Add(BoxStaticDropletWall(evo:LevelSetEvolution.Phasefield, cahnfactor: 2.0, theta: theta));
    controls.Add(BoxStaticDropletWall(evo:LevelSetEvolution.Phasefield, cahnfactor: 3.0, theta: theta));
    controls.Add(BoxStaticDropletWall(evo:LevelSetEvolution.Phasefield, cahnfactor: 4.0, theta: theta));
}

In [ ]:
controls.ForEach(c => Console.WriteLine(c.SessionName));
int ctrlCount = controls.Count();

In [207]:
//// FOR TESTING IF THIS RUNS
////foreach(var C in controls){
//var C = controls.ElementAt(1);
//C.ImmediatePlotPeriod = 1;
//C.SuperSampling = 2;
//C.savetodb = false;
//using(var solver = new XNSE()){
//    solver.Init(C);
//    solver.RunSolverMode();
//}
////}


In [208]:
bool run      = true;
var bpc = BoSSSshell.GetDefaultQueue();
bpc.Display();

In [209]:
//BoSSSshell.WorkflowMgm.ResetProject(true, true, true, true);

In [210]:
if(BoSSSshell.WorkflowMgm.AllJobs.Count() > 0){
     BoSSSshell.WorkflowMgm.ResetProject();
}
var jobs = controls.Select(c => c.CreateJob()).ToArray();
jobs.ForEach(j => j.NumberOfMPIProcs = 1);
jobs.ForEach(j => j.NumberOfThreads = 1);
jobs.ForEach(j => j.EnvironmentVars["BOSSS_ARG_7"] = j.NumberOfThreads.ToString());

In [ ]:
jobs.Activate();

In [ ]:
BoSSSshell.WorkflowMgm.BlockUntilAllJobsTerminate(86400);

In [ ]:
int count = BoSSSshell.wmg.Sessions.Count();
int success = BoSSSshell.wmg.Sessions.Where(s => s.SuccessfulTermination).Count();

if(count != ctrlCount || count != success){
    throw new ApplicationException("Not all simulations calculated or finished successful!");
}